# Lab Assignment 10 - NLP using Python-II
**UCS420 Cognitive Computing**
Feature extraction from text, sentiment analysis and text generation

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

stop_words = set(stopwords.words("english"))

## Q1. Preprocessing and tokenization

In [ ]:
text = ("Street food is my favourite thing about travelling because every city has its own flavours. "
        "Nothing beats a plate of hot samosas with mint chutney on a rainy evening. "
        "I have tried over 20 different snacks across India, and each one tells a story about the people who make it. "
        "Cooking at home is fun too, but it isn't the same as eating from a roadside cart. "
        "Spices, fresh ingredients and a bit of patience are what turn simple recipes into memorable meals. "
        "Sharing food with friends and family always makes me happier than any expensive restaurant could.")
print(text)

**1. Lowercase and remove punctuation using re**

In [ ]:
clean_text = re.sub(r"[^\w\s]", "", text.lower())
print(clean_text)

**2. Tokenize into words and sentences**

In [ ]:
words = word_tokenize(clean_text)
sentences = sent_tokenize(text)

print(words)
print()
for i, s in enumerate(sentences, 1):
    print(i, s)

**3. split() vs word_tokenize()** (run on the original text so the difference is visible)

In [ ]:
split_tokens = text.split()
nltk_tokens = word_tokenize(text)

print("split():       ", len(split_tokens), "tokens")
print("word_tokenize():", len(nltk_tokens), "tokens")

print("\nOnly in split():", [t for t in split_tokens if t not in nltk_tokens])
print("\nOnly in word_tokenize():", [t for t in nltk_tokens if t not in split_tokens])

`split()` only breaks on whitespace, so punctuation stays glued to words (`"flavours."`, `"India,"`) and "isn't" stays as one token. `word_tokenize()` separates punctuation into its own tokens and splits contractions into `is` + `n't`, which is closer to how a language model sees the text.

**4. Remove stopwords**

In [ ]:
filtered_words = [w for w in words if w not in stop_words]
print(filtered_words)

**5. Word frequency distribution**

In [ ]:
freq = FreqDist(filtered_words)
print(freq.most_common(10))

plt.figure(figsize=(10, 5))
freq.plot(15, title="Top 15 words (stopwords removed)")
plt.show()

## Q2. Stemming vs lemmatization on the same paragraph

In [ ]:
alpha_words = re.findall(r"[a-zA-Z]+", text.lower())
alpha_words = [w for w in alpha_words if w not in stop_words]
print(alpha_words)

In [ ]:
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

comparison = pd.DataFrame({
    "word": alpha_words,
    "stemmed": [porter.stem(w) for w in alpha_words],
    "lemmatized": [lemmatizer.lemmatize(w, pos="v") for w in alpha_words],
})
comparison.drop_duplicates("word").reset_index(drop=True)

**Which one to prefer?**

Stemming just chops word endings with fixed rules, so it is fast but the result is often not a real word (for example, "memorable" and "ingredients" get trimmed to stubs). Lemmatization looks words up in WordNet and returns a proper dictionary form, but it is slower and needs the right part of speech to work well.

- Prefer **stemming** for search engines, indexing and quick classification where speed matters and exact word form does not.
- Prefer **lemmatization** for chatbots, sentiment analysis, or anything where the output is read by people or the meaning matters.

## Q3. Bag of Words and TF-IDF

In [ ]:
docs = [
    "The new phone has an amazing camera, the camera app is fast and the battery lasts two days.",
    "Delivery was late, the packaging was damaged and the delivery service was very disappointing.",
    "Government announces new education policy focused on digital education and skill training.",
]

**1. Bag of Words**

In [ ]:
cv = CountVectorizer(stop_words="english")
bow = cv.fit_transform(docs)

bow_df = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=["Text 1", "Text 2", "Text 3"])
bow_df

**2. TF-IDF scores**

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(docs)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out(), index=["Text 1", "Text 2", "Text 3"])
tfidf_df.round(3)

**3. Top 3 keywords per text**

In [ ]:
for name, row in tfidf_df.iterrows():
    top3 = row.sort_values(ascending=False).head(3)
    print(name)
    for word, score in top3.items():
        print(f"   {word:<12} {score:.3f}")

**Interpretation**

TF-IDF gives a high score to words that appear often inside one text but rarely in the others. So "camera" tops the phone review, "delivery" tops the complaint, and "education" tops the policy headline: these are the words that best describe what each text is about. Words shared between texts (like "new") get a lower score because they do not help tell the texts apart. Where two words have the same score, it is just because they occur the same number of times in the same single text.

## Q4. Text similarity

In [ ]:
ai_text = ("Artificial intelligence lets machines learn from data and make decisions without being explicitly programmed. "
           "Machine learning models are trained on large datasets to recognise images, understand language and predict outcomes. "
           "Companies use AI for chatbots, recommendations and medical diagnosis. "
           "The technology keeps improving as more data and computing power become available.")

blockchain_text = ("Blockchain is a distributed ledger that records transactions across many computers. "
                   "Each block is linked to the previous one using cryptography, which makes the data very hard to change. "
                   "It is used for cryptocurrencies, supply chain tracking and secure digital contracts. "
                   "The technology removes the need for a central authority because the network verifies every transaction.")

**1. Preprocess and tokenize**

In [ ]:
lemmatizer = WordNetLemmatizer()

def preprocess(s):
    tokens = word_tokenize(re.sub(r"[^\w\s]", "", s.lower()))
    return [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and t.isalpha()]

ai_tokens = preprocess(ai_text)
bc_tokens = preprocess(blockchain_text)

print("AI:", ai_tokens)
print("\nBlockchain:", bc_tokens)

**2a. Jaccard similarity**

In [ ]:
set_ai, set_bc = set(ai_tokens), set(bc_tokens)

jaccard = len(set_ai & set_bc) / len(set_ai | set_bc)
print("Common words:", set_ai & set_bc)
print(f"Jaccard similarity: {jaccard:.3f}")

**2b. Cosine similarity with TF-IDF**

In [ ]:
vec = TfidfVectorizer()
matrix = vec.fit_transform([" ".join(ai_tokens), " ".join(bc_tokens)])

cos_sim = cosine_similarity(matrix[0], matrix[1])[0][0]
print(f"Cosine similarity: {cos_sim:.3f}")

**2c. Which metric is more useful here?**

Jaccard only asks which words are shared and ignores how often or how importantly they appear, so every word counts the same. Cosine similarity on TF-IDF vectors also takes word frequency and rarity into account, so a shared generic word contributes little, while shared distinctive words matter more. For these two short texts about different technologies, both scores come out low, which is correct. Cosine gives a finer-grained score and is the better choice for longer texts, whereas Jaccard is a quick sanity check that is easy to explain.

## Q5. Sentiment analysis

In [ ]:
from textblob import TextBlob

reviews = [
    "The headphones sound amazing and the battery life is excellent. Totally worth the price!",
    "Worst purchase ever. The product stopped working after two days and support was useless.",
    "Delivery was quick and the packaging was neat. Very happy with the quality.",
    "The laptop is okay. It does the job but nothing special.",
    "Terrible service, rude staff and a long waiting time. I will never come back.",
    "Great phone with a beautiful display and a smooth camera. Love it!",
    "The app crashes all the time and the design looks outdated and confusing.",
    "The restaurant was fine. Food arrived on time and tasted average.",
    "Fantastic customer support, they solved my issue within minutes. Highly recommended.",
]

In [ ]:
def label_sentiment(score):
    if score > 0.1:
        return "Positive"
    if score < -0.1:
        return "Negative"
    return "Neutral"

rows = []
for r in reviews:
    blob = TextBlob(r)
    rows.append({
        "review": r,
        "polarity": round(blob.sentiment.polarity, 3),
        "subjectivity": round(blob.sentiment.subjectivity, 3),
        "label": label_sentiment(blob.sentiment.polarity),
    })

sentiment_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
sentiment_df

In [ ]:
sentiment_df["label"].value_counts()

**Word cloud of positive reviews**

In [ ]:
from wordcloud import WordCloud, STOPWORDS

positive_text = " ".join(sentiment_df.loc[sentiment_df["label"] == "Positive", "review"])

cloud = WordCloud(width=800, height=400, background_color="white", stopwords=STOPWORDS).generate(positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(cloud, interpolation="bilinear")
plt.axis("off")
plt.title("Positive reviews")
plt.show()

## Q6. Text generation with an LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import pad_sequences

try:
    from tensorflow.keras.preprocessing.text import Tokenizer
except ImportError:
    from keras.preprocessing.text import Tokenizer

training_text = ("Learning to code has changed the way I think about problems. "
                 "Every program starts with a small idea and grows step by step. "
                 "I write a few lines, run them, and fix the errors that show up. "
                 "Mistakes are not failures, they are hints that point toward the solution. "
                 "With practice the patterns start to feel familiar and the work becomes enjoyable. "
                 "Good programmers read a lot, ask questions and never stop experimenting. "
                 "I want to build tools that help people save time and learn new things every day. "
                 "Curiosity keeps me going and patience helps me finish what I start.")
print(len(training_text.split()), "words")

**1. Tokenize with Keras Tokenizer**

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

**2. Input sequences and model**

In [ ]:
input_sequences = []
for line in training_text.split("."):
    if not line.strip():
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(2, len(token_list) + 1):
        input_sequences.append(token_list[:i])

max_len = max(len(s) for s in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
print("Training samples:", X.shape[0])

In [ ]:
model = Sequential([
    Embedding(total_words, 32),
    LSTM(100),
    Dense(total_words, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.build(input_shape=(None, max_len - 1))
model.summary()

**3. Train and generate text**

In [ ]:
history = model.fit(X, y, epochs=300, batch_size=16, verbose=0)
print("Final training accuracy:", round(history.history["accuracy"][-1], 3))

In [ ]:
def generate_line(seed, n_words=8):
    line = seed
    for _ in range(n_words):
        seq = tokenizer.texts_to_sequences([line])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding="pre")
        probs = model.predict(seq, verbose=0)[0]
        word = tokenizer.index_word.get(int(np.argmax(probs)))
        if word is None:
            break
        line += " " + word
    return line

for seed in ["learning", "every", "curiosity"]:
    print(generate_line(seed))

The training text is tiny, so the model mostly memorises it and the generated lines closely follow the original sentences. With a bigger corpus the output would be more varied.